
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>


# Create and govern data objects with Unity Catalog

In this notebook you will learn how to:
* Create catalogs, schemas, tables, views and user-defined functions
* Control access to these objects
* Use dynamic views to protect columns and rows within tables
* Explore grants on various objects in Unity Catalog

## Prerequisites

If you would like to follow along with this lab, you must:
* Have metastore admin permissions in order to create and manage a catalog
* Have a SQL warehouse to which the user mentioned above has access
  * See notebook: Creating compute resources for Unity Catalog access

## Setup

Run the following cell to perform some setup. In order to avoid conflicts in a shared training environment, this will generate a unique catalog name exclusively for your use, which we will employ shortly.

In [0]:
%run ./Includes/Classroom-Setup-06.1

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.


Resetting the learning environment:
| No action taken

Skipping install of existing datasets to "dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04"

Validating the locally installed datasets:
| listing local files...(5 seconds)
| validation completed...(5 seconds total)

Using the catalog "eu_west2_space" and the schema "default".

Predefined paths variables:
| DA.paths.working_dir: dbfs:/mnt/dbacademy-users/shifajamali55@gmail.com/data-engineering-with-databricks
| DA.paths.user_db:     dbfs:/mnt/dbacademy-users/shifajamali55@gmail.com/data-engineering-with-databricks/database.db
| DA.paths.datasets:    dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04

Setup completed (8 seconds)


Current Catalog: eu_west2_space
Your Catalog:    shifajamali55_soeb_da


## Unity Catalog's three-level namespace

Anyone with SQL experience will likely be familiar with the traditional two-level namespace to address tables or views within a schema as follows, as shown in the following example query:

    SELECT * FROM myschema.mytable;

Unity Catalog introduces the concept of a **catalog** into the hierarchy. As a container for schemas, the catalog provides a new way for organizations to segregate their data. There can be as many catalogs as you like, which in turn can contain as many schemas as you like (the concept of a **schema** is unchanged by Unity Catalog; schemas contain data objects like tables, views, and user-defined functions).

To deal with this additional level, complete table/view references in Unity Catalog use a three-level namespace. The following query exemplifies this:

    SELECT * FROM mycatalog.myschema.mytable;

This can be handy in many use cases. For example:

* Separating data relating to business units within your organization (sales, marketing, human resources, etc)
* Satisfying SDLC requirements (dev, staging, prod, etc)
* Establishing sandboxes containing temporary datasets for internal use

### Create a new catalog
Let's create a new catalog in our metastore. The variable **`${DA.my_new_catalog}`** was displayed by the setup cell above, containing a unique string generated based on your username.

Run the **`CREATE`** statement below, and click the **Data** icon in the left sidebar to confirm this new catalog was created.

In [0]:
CREATE CATALOG IF NOT EXISTS ${DA.my_new_catalog} 

com.databricks.backend.common.rpc.SparkDriverExceptions$SQLExecutionException: com.databricks.sql.managedcatalog.UnityCatalogServiceException: [RequestId=25755ab6-7654-4bb4-96e1-7d712c3e4522 ErrorClass=INVALID_PARAMETER_VALUE.UNSUPPORTED_LOCATION_SCHEME] storage_root has invalid URI scheme dbfs. Valid URI schemes include s3, gs, s3n, wasbs, r2, s3a, abfss
	at com.databricks.managedcatalog.ErrorDetailsHandler.wrapServiceException(ErrorDetailsHandler.scala:46)
	at com.databricks.managedcatalog.ErrorDetailsHandler.wrapServiceException$(ErrorDetailsHandler.scala:28)
	at com.databricks.managedcatalog.ManagedCatalogClientImpl.wrapServiceException(ManagedCatalogClientImpl.scala:151)
	at com.databricks.managedcatalog.ManagedCatalogClientImpl.recordAndWrapException(ManagedCatalogClientImpl.scala:4795)
	at com.databricks.managedcatalog.ManagedCatalogClientImpl.createCatalog(ManagedCatalogClientImpl.scala:283)
	at com.databricks.sql.managedcatalog.ManagedCatalogCommon.createCatalog(ManagedCatalogCommon.scala:405)
	at com.databricks.sql.managedcatalog.ProfiledManagedCatalog.$anonfun$createCatalog$1(ProfiledManagedCatalog.scala:94)
	at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.java:23)
	at org.apache.spark.sql.catalyst.MetricKeyUtils$.measure(MetricKey.scala:783)
	at com.databricks.sql.managedcatalog.ProfiledManagedCatalog.$anonfun$profile$1(ProfiledManagedCatalog.scala:61)
	at com.databricks.spark.util.FrameProfiler$.record(FrameProfiler.scala:94)
	at com.databricks.sql.managedcatalog.ProfiledManagedCatalog.profile(ProfiledManagedCatalog.scala:60)
	at com.databricks.sql.managedcatalog.ProfiledManagedCatalog.createCatalog(ProfiledManagedCatalog.scala:94)
	at com.databricks.sql.managedcatalog.ManagedCatalogSessionCatalog.createCatalog(ManagedCatalogSessionCatalog.scala:518)
	at com.databricks.sql.managedcatalog.command.CreateCatalogCommand.run(CatalogCommands.scala:46)
	at org.apache.spark.sql.execution.command.ExecutedCommandExec.$anonfun$sideEffectResult$1(commands.scala:82)
	at com.databricks.spark.util.FrameProfiler$.record(FrameProfiler.scala:94)
	at org.apache.spark.sql.execution.command.ExecutedCommandExec.sideEffectResult$lzycompute(commands.scala:80)
	at org.apache.spark.sql.execution.command.ExecutedCommandExec.sideEffectResult(commands.scala:79)
	at org.apache.spark.sql.execution.command.ExecutedCommandExec.executeCollect(commands.scala:91)
	at org.apache.spark.sql.execution.QueryExecution$$anonfun$$nestedInanonfun$eagerlyExecuteCommands$1$1.$anonfun$applyOrElse$3(QueryExecution.scala:286)
	at org.apache.spark.sql.catalyst.QueryPlanningTracker$.withTracker(QueryPlanningTracker.scala:166)
	at org.apache.spark.sql.execution.QueryExecution$$anonfun$$nestedInanonfun$eagerlyExecuteCommands$1$1.$anonfun$applyOrElse$2(QueryExecution.scala:286)
	at org.apache.spark.sql.execution.SQLExecution$.$anonfun$withCustomExecutionEnv$9(SQLExecution.scala:303)
	at org.apache.spark.sql.execution.SQLExecution$.withSQLConfPropagated(SQLExecution.scala:533)
	at org.apache.spark.sql.execution.SQLExecution$.$anonfun$withCustomExecutionEnv$1(SQLExecution.scala:226)
	at org.apache.spark.sql.SparkSession.withActive(SparkSession.scala:1148)
	at org.apache.spark.sql.execution.SQLExecution$.withCustomExecutionEnv(SQLExecution.scala:155)
	at org.apache.spark.sql.execution.SQLExecution$.withNewExecutionId(SQLExecution.scala:482)
	at org.apache.spark.sql.execution.QueryExecution$$anonfun$$nestedInanonfun$eagerlyExecuteCommands$1$1.$anonfun$applyOrElse$1(QueryExecution.scala:285)
	at org.apache.spark.sql.execution.QueryExecution.org$apache$spark$sql$execution$QueryExecution$$withMVTagsIfNecessary(QueryExecution.scala:259)
	at org.apache.spark.sql.execution.QueryExecution$$anonfun$$nestedInanonfun$eagerlyExecuteCommands$1$1.applyOrElse(QueryExecution.scala:280)
	at org.apache.spark.sql.execution.QueryExecution$$anonfun$$nestedInanonfun$eagerlyExecuteCommands$1$1.applyOrElse(QueryExecution.scala:265)
	at org.apache.spark.sql.catalyst.trees.TreeNo

### Select a default catalog

SQL developers will probably also be familiar with the **`USE`** statement to select a default schema, thereby shortening queries by not having to specify it all the time. To extend this convenience while dealing with the extra level in the namespace, Unity Catalog augments the language with two additional statements, shown in the examples below:

    USE CATALOG mycatalog;
    USE SCHEMA myschema;  
    
Let's select the newly created catalog as the default. Now, any schema references will be assumed to be in this catalog unless explicitly overridden by a catalog reference.

In [0]:
SHOW CATALOGS;

catalog
eu_west2_space
hive_metastore
samples
system


In [0]:
USE CATALOG eu_west2_space;

### Create and use a new schema
Next, let's create a schema in this new catalog. We won't need to generate another unique name for this schema, since we're now using a unique catalog that is isolated from the rest of the metastore. Let's also set this as the default schema. Now, any data references will be assumed to be in the catalog and schema we created, unless explicitely overridden by a two- or three-level reference.

Run the code below, and click the **Data** icon in the left sidebar to confirm this schema was created in the new catalog we created.

In [0]:
CREATE SCHEMA IF NOT EXISTS example;
USE SCHEMA example

### Set up tables and views

With all the necessary containment in place, let's set up tables and views. For this example, we'll use mock data to create and populate a *silver* managed table with synthetic patient heart rate data and a *gold* view that averages heart rate data per patient on a daily basis.

Run the cells below, and click the **Data** icon in the left sidebar to explore the contents of the *example* schema. Note that we don't need to specify three levels when specifying the table or view names below, since we selected a default catalog and schema.

In [0]:
-- Creating the silver table with patient heart rate data and PII

CREATE OR REPLACE TABLE heartrate_device (device_id INT, mrn STRING, name STRING, time TIMESTAMP, heartrate DOUBLE);

INSERT INTO heartrate_device VALUES
  (23, "40580129", "Nicholas Spears", "2020-02-01T00:01:58.000+0000", 54.0122153343),
  (17, "52804177", "Lynn Russell", "2020-02-01T00:02:55.000+0000", 92.5136468131),
  (37, "65300842", "Samuel Hughes", "2020-02-01T00:08:58.000+0000", 52.1354807863),
  (23, "40580129", "Nicholas Spears", "2020-02-01T00:16:51.000+0000", 54.6477014191),
  (17, "52804177", "Lynn Russell", "2020-02-01T00:18:08.000+0000", 95.033344842);
  
SELECT * FROM heartrate_device

device_id,mrn,name,time,heartrate
23,40580129,Nicholas Spears,2020-02-01T00:01:58Z,54.0122153343
17,52804177,Lynn Russell,2020-02-01T00:02:55Z,92.5136468131
37,65300842,Samuel Hughes,2020-02-01T00:08:58Z,52.1354807863
23,40580129,Nicholas Spears,2020-02-01T00:16:51Z,54.6477014191
17,52804177,Lynn Russell,2020-02-01T00:18:08Z,95.033344842


In [0]:
-- Creating a gold view to share with other users

CREATE OR REPLACE VIEW agg_heartrate AS (
  SELECT mrn, name, MEAN(heartrate) avg_heartrate, DATE_TRUNC("DD", time) date
  FROM heartrate_device
  GROUP BY mrn, name, DATE_TRUNC("DD", time)
);
SELECT * FROM agg_heartrate

mrn,name,avg_heartrate,date
52804177,Lynn Russell,93.77349582755,2020-02-01T00:00:00Z
65300842,Samuel Hughes,52.1354807863,2020-02-01T00:00:00Z
40580129,Nicholas Spears,54.329958376700006,2020-02-01T00:00:00Z


Querying the table above works as expected since we are the data owner. That is, we have ownership of the data object we're querying. Querying the view also works because we are the owner of both the view and the table it's referencing. Thus, no object-level permissions are required to access these resources.

## The _account users_ Group

In accounts with Unity Catalog enabled, there is an _account users_ group. This group contains all users that have been assigned to the workspace from the Databricks account. We are going to use this group to show how data object access can be different for users in different groups.


## Grant access to data objects

Unity Catalog employs an explicit permission model by default; no permissions are implied or inherited from containing elements. Therefore, in order to access any data objects, users will need **USAGE** permission on all containing elements; that is, the containing schema and catalog.

Now let's allow members of the *account users* group to query the *gold* view. In order to do this, we need to grant the following permissions:
1. USAGE on the catalog and schema
1. SELECT on the data object (e.g. view)

In [0]:
GRANT USAGE ON CATALOG ${DA.my_new_catalog} TO `account users`;

com.databricks.backend.common.rpc.SparkDriverExceptions$SQLExecutionException: com.databricks.sql.managedcatalog.UnityCatalogServiceException: [RequestId=c37b314a-d43f-402d-9ed3-1b38587ec9de ErrorClass=CATALOG_DOES_NOT_EXIST.RESOURCE_DOES_NOT_EXIST] Catalog 'shifajamali55_soeb_da' does not exist.
	at com.databricks.managedcatalog.ErrorDetailsHandler.wrapServiceException(ErrorDetailsHandler.scala:46)
	at com.databricks.managedcatalog.ErrorDetailsHandler.wrapServiceException$(ErrorDetailsHandler.scala:28)
	at com.databricks.managedcatalog.ManagedCatalogClientImpl.wrapServiceException(ManagedCatalogClientImpl.scala:151)
	at com.databricks.managedcatalog.ManagedCatalogClientImpl.recordAndWrapException(ManagedCatalogClientImpl.scala:4795)
	at com.databricks.managedcatalog.ManagedCatalogClientImpl.updatePermissions(ManagedCatalogClientImpl.scala:3268)
	at com.databricks.sql.managedcatalog.ManagedCatalogCommon.addPermissions(ManagedCatalogCommon.scala:2360)
	at com.databricks.sql.managedcatalog.ProfiledManagedCatalog.$anonfun$addPermissions$1(ProfiledManagedCatalog.scala:406)
	at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.java:23)
	at org.apache.spark.sql.catalyst.MetricKeyUtils$.measure(MetricKey.scala:783)
	at com.databricks.sql.managedcatalog.ProfiledManagedCatalog.$anonfun$profile$1(ProfiledManagedCatalog.scala:61)
	at com.databricks.spark.util.FrameProfiler$.record(FrameProfiler.scala:94)
	at com.databricks.sql.managedcatalog.ProfiledManagedCatalog.profile(ProfiledManagedCatalog.scala:60)
	at com.databricks.sql.managedcatalog.ProfiledManagedCatalog.addPermissions(ProfiledManagedCatalog.scala:406)
	at com.databricks.sql.managedcatalog.command.GrantPermissionsCommandV2.run(PermissionCommands.scala:125)
	at org.apache.spark.sql.execution.command.ExecutedCommandExec.$anonfun$sideEffectResult$1(commands.scala:82)
	at com.databricks.spark.util.FrameProfiler$.record(FrameProfiler.scala:94)
	at org.apache.spark.sql.execution.command.ExecutedCommandExec.sideEffectResult$lzycompute(commands.scala:80)
	at org.apache.spark.sql.execution.command.ExecutedCommandExec.sideEffectResult(commands.scala:79)
	at org.apache.spark.sql.execution.command.ExecutedCommandExec.executeCollect(commands.scala:91)
	at org.apache.spark.sql.execution.QueryExecution$$anonfun$$nestedInanonfun$eagerlyExecuteCommands$1$1.$anonfun$applyOrElse$3(QueryExecution.scala:286)
	at org.apache.spark.sql.catalyst.QueryPlanningTracker$.withTracker(QueryPlanningTracker.scala:166)
	at org.apache.spark.sql.execution.QueryExecution$$anonfun$$nestedInanonfun$eagerlyExecuteCommands$1$1.$anonfun$applyOrElse$2(QueryExecution.scala:286)
	at org.apache.spark.sql.execution.SQLExecution$.$anonfun$withCustomExecutionEnv$9(SQLExecution.scala:303)
	at org.apache.spark.sql.execution.SQLExecution$.withSQLConfPropagated(SQLExecution.scala:533)
	at org.apache.spark.sql.execution.SQLExecution$.$anonfun$withCustomExecutionEnv$1(SQLExecution.scala:226)
	at org.apache.spark.sql.SparkSession.withActive(SparkSession.scala:1148)
	at org.apache.spark.sql.execution.SQLExecution$.withCustomExecutionEnv(SQLExecution.scala:155)
	at org.apache.spark.sql.execution.SQLExecution$.withNewExecutionId(SQLExecution.scala:482)
	at org.apache.spark.sql.execution.QueryExecution$$anonfun$$nestedInanonfun$eagerlyExecuteCommands$1$1.$anonfun$applyOrElse$1(QueryExecution.scala:285)
	at org.apache.spark.sql.execution.QueryExecution.org$apache$spark$sql$execution$QueryExecution$$withMVTagsIfNecessary(QueryExecution.scala:259)
	at org.apache.spark.sql.execution.QueryExecution$$anonfun$$nestedInanonfun$eagerlyExecuteCommands$1$1.applyOrElse(QueryExecution.scala:280)
	at org.apache.spark.sql.execution.QueryExecution$$anonfun$$nestedInanonfun$eagerlyExecuteCommands$1$1.applyOrElse(QueryExecution.scala:265)
	at org.apache.spark.sql.catalyst.trees.TreeNode.$anonfun$transformDownWithPruning$1(TreeNode.scala:465)
	at org.apache.spark.sql.catalyst.trees.CurrentOrigin$.withOrigin(origin.scala:69)
	at org.apache.spar

In [0]:
GRANT USAGE ON SCHEMA example TO `account users`;

In [0]:
GRANT SELECT ON VIEW agg_heartrate to `account users`

### Query the view

With a data object hierarchy and all the appropriate grants in place, let's attempt to perform a query on the *gold* view.

All of us are members of the **account users** group, so can use this group to verify our configuration, and observe the impact when we make changes.

1. In the upper-left corner, click the app switcher to open it up.
1. Right-click on **SQL**, and select **Open Link in New Tab**.
1. Go to the **Queries** page and click **Create query**.
1. Select the shared SQL warehouse that was created while following the *Creating compute resources for Unity Catalog access* demo.
1. Return to this notebook and continue following along. When prompted, we will be switching to the Databricks SQL session and executing queries.

The following cell generates a fully qualified query statement that specifies all three levels for the view, since we will be running this in an environment that doesn't have variables or a default catalog and schema set up. Run the query generated below in the Databricks SQL session. Since all appropriate grants are in place for account users to access the view, the output should resemble what we saw earlier when querying the *gold* view.

In [0]:
SELECT "SELECT * FROM ${DA.my_new_catalog}.example.agg_heartrate" AS Query

Query
SELECT * FROM shifajamali55_soeb_da.example.agg_heartrate


### Query the silver table
Back in the same query in the Databricks SQL session, let's replace *gold* with *silver* and run the query. As the data owner, running this should return all table results as expected. However, if you were to run this as another user, this would fail because we never set up permissions on the *silver* table. 

Querying *gold* works for all members of the _account users_ group because the query represented by a view is essentially executed as the owner of the view. This important property enables some interesting security use cases; in this way, views can provide users with a restricted view of sensitive data, without providing access to the underlying data itself. We will see more of this shortly.

For now, you can close and discard the *silver* query in the Databricks SQL session; we will not be using it any more.

### Create and grant access to a user-defined function

Unity Catalog is capable of managing user-defined functions within schemas as well. The code below sets up a simple function that masks all but the last two characters of a string, and then tries it out. Once again, we are the data owner so no grants are required.

In [0]:
CREATE OR REPLACE FUNCTION my_mask(x STRING)
  RETURNS STRING
  RETURN CONCAT(REPEAT("*", LENGTH(x) - 2), RIGHT(x, 2)
); 
SELECT my_mask('sensitive data') AS data

data
************ta



To allow members of the *account users* group to run our function, they need **EXECUTE** on the function, along with the requisite **USAGE** grants on the schema and catalog that we've mentioned before.

In [0]:
GRANT EXECUTE ON FUNCTION my_mask to `account users`

### Run a function

Now we'll try the function in Databricks SQL. Paste the fully qualified query statement generated below into a new query to run this function in Databricks SQL. Since all appropriate grants are in place to access the function, the output should resemble what we just saw above.

In [0]:
SELECT "SELECT ${DA.my_new_catalog}.example.my_mask('sensitive data') AS data" AS Query

Query
SELECT shifajamali55_soeb_da.example.my_mask('sensitive data') AS data


## Protect table columns and rows with dynamic views

We have seen that Unity Catalog's treatment of views provides the ability for views to protect access to tables; users can be granted access to views that manipulate, transform, or obscure data from a source table, without needing to provide direct access to the source table.

Dynamic views provide the ability to do fine-grained access control of columns and rows within a table, conditional on the principal running the query. Dynamic views are an extension to standard views that allow us to do things like:
* partially obscure column values or redact them entirely
* omit rows based on specific criteria

Access control with dynamic views is achieved through the use of functions within the definition of the view. These functions include:
* **`current_user()`**: returns the email address of the user querying the view
* **`is_account_group_member()`**: returns TRUE if the user querying the view is a member of the specified group

Note: please refrain from using the legacy function **`is_member()`**, which references workspace-level groups. This is not good practice in the context of Unity Catalog.

### Redact columns

Suppose we want account users to be able to see aggregated data trends from the *gold* view, but we don't want to disclose patient PII. Let's redefine the view to redact the *mrn* and *name* columns using the **`is_account_group_member()`**.

Note: this is a simple training example that doesn't necessarily align with general best practices. For a production system, a more secure approach would be to redact column values for all users who are *not* members of a specific group.


In [0]:
CREATE OR REPLACE VIEW agg_heartrate AS
SELECT
  CASE WHEN
    is_account_group_member('account users') THEN 'REDACTED'
    ELSE mrn
  END AS mrn,
  CASE WHEN
    is_account_group_member('account users') THEN 'REDACTED'
    ELSE name
  END AS name,
  MEAN(heartrate) avg_heartrate,
  DATE_TRUNC("DD", time) date
  FROM heartrate_device
  GROUP BY mrn, name, DATE_TRUNC("DD", time)


Re-issue the grant.

In [0]:
GRANT SELECT ON VIEW agg_heartrate to `account users`


Now, revisit Databricks SQL and rerun the query on the *gold* view. Run the cell below to generate this query. 

We will see that the *mrn* and *name* column values have been redacted.

In [0]:
SELECT "SELECT * FROM ${DA.my_new_catalog}.example.agg_heartrate" AS Query

Query
SELECT * FROM shifajamali55_soeb_da.example.agg_heartrate


### Restrict rows

Now let's suppose we want a view that, rather than aggregating and redacting columns, simply filters out rows from the source. Let's  apply the same **`is_account_group_member()`** function to create a view that passes through only rows whose *device_id* is less than 30. Row filtering is done by applying the conditional as a **`WHERE`** clause.

In [0]:
CREATE OR REPLACE VIEW agg_heartrate AS
SELECT
  mrn,
  time,
  device_id,
  heartrate
FROM heartrate_device
WHERE
  CASE WHEN
    is_account_group_member('account users') THEN device_id < 30
    ELSE TRUE
  END


Re-issue the grant.

In [0]:
GRANT SELECT ON VIEW agg_heartrate to `account users`


For any user who is not part of the group, querying the view above displays all five records. Now, revisit Databricks SQL and rerun the query on the *gold* view. We will see that one of the records is missing. The missing record contained a value for *device_id* that was caught by the filter.

### Data masking
One final use case for dynamic views is data masking, or partially obscuring data. In the first example, we redacted columns entirely. Masking is similar in principle except we are displaying some of the data rather than replacing it entirely. And for this simple example, we'll leverage the *my_mask()* user-defined function that we created earlier to mask the *mrn* column, though SQL provides a fairly comprehensive library of built-in data manipulation functions that can be leveraged to mask data in a number of different ways. It's good practice to leverage those when you can.

In [0]:
DROP VIEW IF EXISTS agg_heartrate;

CREATE VIEW agg_heartrate AS
SELECT
  CASE WHEN
    is_account_group_member('account users') THEN my_mask(mrn)
    ELSE mrn
  END AS mrn,
  time,
  device_id,
  heartrate
FROM heartrate_device
WHERE
  CASE WHEN
    is_account_group_member('account users') THEN device_id < 30
    ELSE TRUE
  END


Re-issue the grant.

In [0]:
GRANT SELECT ON VIEW agg_heartrate to `account users`


For any user not a member of the group, this displays undisturbed records. Revisit Databricks SQL and rerun the query on the *gold* view. All values in the *mrn* column will be masked.


## Explore objects

Let's explore some SQL statements to examine our data objects and permissions. Let's begin by taking stock of the objects we have in the *examples* schema.

In [0]:
SHOW TABLES

database,tableName,isTemporary
example,agg_heartrate,false
example,heartrate_device,false


In [0]:
SHOW VIEWS

namespace,viewName,isTemporary,isMaterialized
example,agg_heartrate,false,false


In the above two statements, we didn't specify a schema since we are relying on the defaults we selected. Alternatively, we could have been more explicit using a statement like **`SHOW TABLES IN example`**.

Now let's step up a level in the hierarchy and take inventory of the schemas in our catalog. Once again, we are leveraging the fact that we have a default catalog selected. If we wanted to be more explicit, we could use something like **`SHOW SCHEMAS IN ${DA.my_new_catalog}`**.

In [0]:
SHOW SCHEMAS

databaseName
default
example
information_schema
tutorial


The *example* schema, of course, is the one we created earlier. The *default* schema is created by default as per SQL conventions when creating a new catalog.

Finally, let's list the catalogs in our metastore.

In [0]:
SHOW CATALOGS

catalog
eu_west2_space
hive_metastore
samples
system


There may be more entries than you were expecting. At a minimum, you will see:
* A catalog beginning with the prefix *dbacademy_*, which is the one we created earlier.
* *hive_metastore*, which is not a real catalog in the metastore, but rather a virtual representation of the workspace local Hive metastore. Use this to access workspace-local tables and views.
* *main*, a catalog which is created by default with each new metastore.
* *samples*, another virtual catalog that presents example datasets provided by Databricks

There may be more catalogs present depending on the historical activity in your metastore.

### Explore permissions

Now let's explore permissions using **`SHOW GRANTS`**, starting with the *gold* view and working our way up.

In [0]:
SHOW GRANTS ON VIEW agg_heartrate

Principal,ActionType,ObjectType,ObjectKey
account users,SELECT,TABLE,eu_west2_space.example.agg_heartrate


Currenly there is only the **SELECT** grant that we just set up. Now let's check the grants on *silver*.

In [0]:
SHOW GRANTS ON TABLE heartrate_device

Principal,ActionType,ObjectType,ObjectKey


There are no grants on this table currently. Only we, the data owner, can access this table directly. Anyone with permission to access the *gold* view, for which we are also the data owner, has the ability to access this table indirectly.

Now let's look at the containing schema.

In [0]:
SHOW GRANTS ON SCHEMA example

Principal,ActionType,ObjectType,ObjectKey


Currently we see the **USAGE** grant we set up earlier.

Now let's examine the catalog.

In [0]:
SHOW GRANTS ON CATALOG ${DA.my_new_catalog}

com.databricks.backend.common.rpc.SparkDriverExceptions$SQLExecutionException: com.databricks.sql.managedcatalog.UnityCatalogServiceException: [RequestId=36e415c4-76e4-42fb-a597-c06656db5fd0 ErrorClass=CATALOG_DOES_NOT_EXIST.RESOURCE_DOES_NOT_EXIST] Catalog 'shifajamali55_soeb_da' does not exist.
	at com.databricks.managedcatalog.ErrorDetailsHandler.wrapServiceException(ErrorDetailsHandler.scala:46)
	at com.databricks.managedcatalog.ErrorDetailsHandler.wrapServiceException$(ErrorDetailsHandler.scala:28)
	at com.databricks.managedcatalog.ManagedCatalogClientImpl.wrapServiceException(ManagedCatalogClientImpl.scala:151)
	at com.databricks.managedcatalog.ManagedCatalogClientImpl.recordAndWrapException(ManagedCatalogClientImpl.scala:4795)
	at com.databricks.managedcatalog.ManagedCatalogClientImpl.getEffectivePermissions(ManagedCatalogClientImpl.scala:3237)
	at com.databricks.sql.managedcatalog.ManagedCatalogCommon.getEffectivePermissions(ManagedCatalogCommon.scala:2354)
	at com.databricks.sql.managedcatalog.ProfiledManagedCatalog.$anonfun$getEffectivePermissions$1(ProfiledManagedCatalog.scala:401)
	at org.apache.spark.sql.catalyst.MetricKeyUtils$.measure(MetricKey.scala:783)
	at com.databricks.sql.managedcatalog.ProfiledManagedCatalog.$anonfun$profile$1(ProfiledManagedCatalog.scala:61)
	at com.databricks.spark.util.FrameProfiler$.record(FrameProfiler.scala:94)
	at com.databricks.sql.managedcatalog.ProfiledManagedCatalog.profile(ProfiledManagedCatalog.scala:60)
	at com.databricks.sql.managedcatalog.ProfiledManagedCatalog.getEffectivePermissions(ProfiledManagedCatalog.scala:401)
	at com.databricks.sql.managedcatalog.command.ShowPermissionsCommandV2.run(PermissionCommands.scala:102)
	at org.apache.spark.sql.execution.command.ExecutedCommandExec.$anonfun$sideEffectResult$1(commands.scala:82)
	at com.databricks.spark.util.FrameProfiler$.record(FrameProfiler.scala:94)
	at org.apache.spark.sql.execution.command.ExecutedCommandExec.sideEffectResult$lzycompute(commands.scala:80)
	at org.apache.spark.sql.execution.command.ExecutedCommandExec.sideEffectResult(commands.scala:79)
	at org.apache.spark.sql.execution.command.ExecutedCommandExec.executeCollect(commands.scala:91)
	at org.apache.spark.sql.execution.QueryExecution$$anonfun$$nestedInanonfun$eagerlyExecuteCommands$1$1.$anonfun$applyOrElse$3(QueryExecution.scala:286)
	at org.apache.spark.sql.catalyst.QueryPlanningTracker$.withTracker(QueryPlanningTracker.scala:166)
	at org.apache.spark.sql.execution.QueryExecution$$anonfun$$nestedInanonfun$eagerlyExecuteCommands$1$1.$anonfun$applyOrElse$2(QueryExecution.scala:286)
	at org.apache.spark.sql.execution.SQLExecution$.$anonfun$withCustomExecutionEnv$9(SQLExecution.scala:303)
	at org.apache.spark.sql.execution.SQLExecution$.withSQLConfPropagated(SQLExecution.scala:533)
	at org.apache.spark.sql.execution.SQLExecution$.$anonfun$withCustomExecutionEnv$1(SQLExecution.scala:226)
	at org.apache.spark.sql.SparkSession.withActive(SparkSession.scala:1148)
	at org.apache.spark.sql.execution.SQLExecution$.withCustomExecutionEnv(SQLExecution.scala:155)
	at org.apache.spark.sql.execution.SQLExecution$.withNewExecutionId(SQLExecution.scala:482)
	at org.apache.spark.sql.execution.QueryExecution$$anonfun$$nestedInanonfun$eagerlyExecuteCommands$1$1.$anonfun$applyOrElse$1(QueryExecution.scala:285)
	at org.apache.spark.sql.execution.QueryExecution.org$apache$spark$sql$execution$QueryExecution$$withMVTagsIfNecessary(QueryExecution.scala:259)
	at org.apache.spark.sql.execution.QueryExecution$$anonfun$$nestedInanonfun$eagerlyExecuteCommands$1$1.applyOrElse(QueryExecution.scala:280)
	at org.apache.spark.sql.execution.QueryExecution$$anonfun$$nestedInanonfun$eagerlyExecuteCommands$1$1.applyOrElse(QueryExecution.scala:265)
	at org.apache.spark.sql.catalyst.trees.TreeNode.$anonfun$transformDownWithPruning$1(TreeNode.scala:465)
	at org.apache.spark.sql.catalyst.trees.CurrentOrigin$.withOrigin(origin.scala:69)
	at org.apache.spark.sql.catalyst.trees.TreeNode.transformDown

Likewise, we see **USAGE** which we granted moments ago.

## Revoke access

No data governance platform would be complete without the ability to revoke previously issued grants. Let's start by examining access to the *my_mask()* function.

In [0]:
SHOW GRANTS ON FUNCTION my_mask

Principal,ActionType,ObjectType,ObjectKey


Now let's revoke this grant.

In [0]:
REVOKE EXECUTE ON FUNCTION my_mask FROM `account users`

Now let's re-examine the access, which will now be empty.

In [0]:
SHOW GRANTS ON FUNCTION my_mask

Principal,ActionType,ObjectType,ObjectKey



Revisit the Databricks SQL session an re-run the query against the *gold* view. Notice that this still works as it did before. Does this surprise you? Why or why not?

Remember that the view is effectively running as its owner, who also happens to own the function and the source table. Just like our view example didn't require direct access to the table being queried since the view owner has ownership of the table, the function continues to work for the same reason.

Now let's try something different. Let's break the permission chain by revoking **USAGE** on the catalog.

In [0]:
REVOKE USAGE ON CATALOG ${DA.my_new_catalog} FROM `account users`


Back in Databricks SQL, re-run the *gold* query, and we see now that even though we have proper permissions on the view and schema, the missing privilege higher up in the hierarchy will break access to this resource. This illustrates Unity Catalog's explicit permission model in action: no permissions are implied or inherited.

## Clean up
Let's run the following cell to remove the catalog that we created earlier. The **`CASCADE`** qualifier will remove the catalog along with any contained elements.

In [0]:
USE CATALOG hive_metastore;
DROP CATALOG IF EXISTS ${DA.my_new_catalog} CASCADE;

In [0]:
%python
DA.cleanup()

Resetting the learning environment:
| No action taken

Validating the locally installed datasets:
| listing local files...(6 seconds)
| validation completed...(6 seconds total)



&copy; 2024 Databricks, Inc. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the 
<a href="https://www.apache.org/">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use">Terms of Use</a> | 
<a href="https://help.databricks.com/">Support</a>